# 01 Corpus and Stylometry

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn
!pip install -q spacy textstat ebooklib beautifulsoup4
!python -m spacy download en_core_web_sm

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

import spacy
import textstat

from ebooklib import epub
from bs4 import BeautifulSoup

nlp = spacy.load("en_core_web_sm")

In [ ]:
from google.colab import files
uploaded = files.upload()

# Upload:
# pg1342-images-3.epub
# pg1400-images-3.epub

In [ ]:
def extract_epub_text(epub_path):

    book = epub.read_epub(epub_path)

    text = []

    for item in book.get_items():

        try:

            soup = BeautifulSoup(
                item.get_content(),
                "html.parser"
            )

            extracted = soup.get_text()

            if len(extracted) > 100:
                text.append(extracted)

        except:
            pass

    return "\n".join(text)

In [ ]:
def clean_text(text):

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(
        r'Project Gutenberg.*',
        '',
        text
    )

    return text.strip()

In [ ]:
def chunk_text(
    text,
    chunk_size=150
):

    words = text.split()

    chunks = []

    for i in range(
        0,
        len(words),
        chunk_size
    ):

        chunk = words[i:i+chunk_size]

        if len(chunk) >= 100:

            chunks.append(
                " ".join(chunk)
            )

    return chunks

In [ ]:
def build_dataset(path, author):

    raw = extract_epub_text(path)

    print(author, "RAW LENGTH =", len(raw))

    clean = clean_text(raw)

    chunks = chunk_text(clean)

    print(author, "CHUNKS =", len(chunks))

    return pd.DataFrame({

        "text": chunks,

        "author": author
    })

In [ ]:
austen_df = build_dataset(
    "pg1342-images-3.epub",
    "Austen"
)

dickens_df = build_dataset(
    "pg1400-images-3.epub",
    "Dickens"
)

human_df = pd.concat([
    austen_df,
    dickens_df
])

print(human_df.shape)
human_df.head()

In [ ]:
def type_token_ratio(text):

    tokens = text.split()

    return len(set(tokens)) / len(tokens)


def hapax_legomena(text):

    tokens = text.split()

    counts = Counter(tokens)

    return sum(
        1
        for _, c in counts.items()
        if c == 1
    )


def avg_sentence_length(text):

    doc = nlp(text)

    sents = list(doc.sents)

    return np.mean([
        len(sent.text.split())
        for sent in sents
    ])


def readability(text):

    return textstat.flesch_kincaid_grade(text)


def extract_features(text):

    return {

        "ttr":
            type_token_ratio(text),

        "hapax":
            hapax_legomena(text),

        "avg_sent_len":
            avg_sentence_length(text),

        "readability":
            readability(text),

        "semicolon":
            text.count(";"),

        "exclamation":
            text.count("!")
    }

In [ ]:
feature_rows = []

for text in human_df["text"]:

    feature_rows.append(
        extract_features(text)
    )

feature_df = pd.DataFrame(
    feature_rows
)

print(feature_df.head())
print(feature_df.shape)

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    feature_df["avg_sent_len"],
    kde=True
)

plt.title(
    "Sentence Length Distribution"
)

plt.show()